<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Computer Vision - Convolutional Neural Network (CNN) - MNIST
</b></font> </br></p>

---

# 0  | Install & Import
***

In [ ]:
# Install

In [ ]:
# Daten & Strukturen
from pandas import read_csv, DataFrame, concat  # CSV laden und Daten zusammenführen
import numpy as np  # Numerische Operationen

# Evaluation & Metriken
from sklearn.metrics import accuracy_score  # Genauigkeit

# Deep Learning Framework
import tensorflow as tf  # TensorFlow Backend
from tensorflow import keras  # Keras API über TensorFlow
import keras  # Keras (Standalone/alias)

# Modell & Architektur
from keras.models import Sequential, load_model  # Modell erstellen und laden
from keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Flatten, Dense  # CNN-Architektur

# Reproduzierbarkeit & Utilities
from keras.utils import set_random_seed, plot_model  # Seed setzen und Modell visualisieren
from tensorflow.config.experimental import enable_op_determinism  # Deterministische Ausführung

# Visualisierung
import matplotlib.pyplot as plt  # Klassische Visualisierung
import plotly.express as px  # Interaktive Plots
import plotly.subplots as sp  # Subplots erstellen

In [ ]:
# Warnung ausstellen
import warnings
warnings.filterwarnings("ignore")

# 1 | Understand
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Aufgabe verstehen</br>
✅ Daten sammeln</br>
✅ Statistische Analyse (Min, Max, Mean, Korrelation, ...)</br>
✅ Datenvisualisierung (Streudiagramm, Box-Plot, ...)</br>
✅ Prepare Schritte festlegen</br>

<p><font color='black' size="5">
Anwendungsfall
</font></p>

---


Die MNIST-Datenbank ( Modified National Institute of Standards and Technology database ) ist eine große Datenbank mit handgeschriebenen Ziffern, die üblicherweise zum Trainieren verschiedener Bildverarbeitungssysteme verwendet wird.

Die MNIST-Datenbank mit handgeschriebenen Ziffern hat einen Trainingssatz von 60.000 Beispielen und einen Testsatz von 10.000 Beispielen mit jeweils  28x28 Pixel.







In [ ]:
(data_train, target_train), (data_test, target_test) = keras.datasets.mnist.load_data()

In [ ]:
data_train.shape, target_train.shape, data_test.shape, target_test.shape


<p><font color='black' size="5">
Anzeigen eines Bildes als Matrix der Pixelwerte
</font></p>

In [ ]:
test_index = 333
image = DataFrame(data_train[test_index])

In [ ]:
image.style.set_properties(**{"font-size": "6pt"}).background_gradient("Greys")

In [ ]:
plt.imshow(data_train[test_index].reshape((28, 28)))

<p><font color='black' size="5">
Anzeigen der ersten 25 Bilder
</font></p>

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(data_train[i].reshape((28, 28)))
    plt.title("Digit:{}".format(target_train[i]))
    label_index = int(target_train[i])
plt.show()

# 2 | Prepare
---


<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Datentyp ermitteln/ändern</br>
✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>

<p><font color='black' size="5">
Skalieren der Daten (0,1)
</font></p>

In [ ]:
data_train = data_train.astype("float32") / 255
data_test = data_test.astype("float32") / 255

<p><font color='black' size="5">
Modellparameter Bildwerte
</font></p>

In [ ]:
num_classes = 10
input_shape = (28, 28, 1)

<p><font color='black' size="5">
Konvertierung einer Klasse (Ganzzahlen 0-9) in einen binären Klassenvektor (0, 1)
</font></p>

In [ ]:
target_train_ = keras.utils.to_categorical(target_train, num_classes)
target_test_ = keras.utils.to_categorical(target_test, num_classes)

# 3 | Modeling
---

<p><font color='black' size="5">
Modellierung eines Neuronalen Netzes
</font></p>

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Cross-Valdiation</br>
✅ Bootstrapping</br>
✅ Regularization</br>

<p><font color='black' size="5">🖼️ Algorithmus: CNN – Convolutional Neural Network (Bildklassifikation)</font></p>

Ein CNN (Convolutional Neural Network) ist ein neuronales Netz, das speziell für Bilddaten entwickelt wurde. Faltungsschichten (Conv2D) erkennen automatisch lokale Muster wie Kanten, Ecken und Formen. Pooling-Schichten reduzieren die Datenmenge, bevor vollständig verbundene Schichten die finale Klassifikation vornehmen.

**Analogie:** Wie das menschliche Sehen: zunächst werden einfache Muster (Linien) erkannt, dann komplexere (Formen), schließlich ganze Objekte.

**Wichtige Schichten:**

| Schicht | Aufgabe |
|---------|--------|
| `Conv2D` | Merkmale im Bild erkennen (Filter) |
| `MaxPooling2D` | Datenmenge reduzieren, robuste Merkmale behalten |
| `Flatten` | 2D-Merkmalskarte in 1D-Vektor umwandeln |
| `Dense` | Klassifikation auf Basis erkannter Merkmale |

In [ ]:
#@markdown   <p><font size="4" color='green'>  CNN – Architektur</font> </br></p>

import base64
from IPython.display import Image, display

diagram = """
flowchart LR
    IMG[Bild] --> CONV[Conv2D - Muster erkennen]
    CONV --> POOL[MaxPooling - Reduzieren]
    POOL --> FLAT[Flatten - 2D zu 1D]
    FLAT --> DENSE[Dense - Klassifikation]
    DENSE --> OUT[Klasse]
"""

encoded = base64.urlsafe_b64encode(diagram.strip().encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}", width=950))

<p><font color='black' size="5">
Zufallszahl initialisieren
</font></p>

In [ ]:
set_random_seed(42)
enable_op_determinism()

<p><font color='black' size="5">
Modellaufbau
</font></p>

In [ ]:
model = Sequential()
model.add(Input(shape=input_shape))
model.add(Conv2D(32, kernel_size=(3, 3), activation="relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation="softmax"))

<p><font color='black' size="5">
Einfaches Layer-Diagramm
</font></p>

In [ ]:
model.summary()

In [ ]:
# Visualisierung neuronales Netz
plot_model(
    model,
    to_file="nn_structure.png",
    show_shapes=True,
    show_dtype=True,
    show_layer_names=True,
    dpi=100,
    expand_nested=True,
    show_layer_activations=True
)

In [ ]:
# Anzahl Parameter je Layer
for layer in model.layers:
    print(f"{layer.name}: {layer.count_params()} Parameter")

<p><font color='black' size="5">
Training
</font></p>


In [ ]:
epochs_ = 10
batch_size_ = 256
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(
    data_train,
    target_train_,
    batch_size=batch_size_,
    epochs=epochs_,
    validation_split=0.2,
)

In [ ]:
print(model.history.params)
print(model.history.history.keys())

<p><font color='black' size="5">
Loss-Entwickung
</font></p>

In [ ]:
import plotly.express as px

title_ = "Loss-Entwicklung"
px.line(
    y=model.history.history["loss"],
    title=title_,
    labels={"x": "Epochen", "y": "Loss-Wert"},
    width=800,
    height=400,
)

# 4 | Evaluate
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Prognose (Train, Test) erstellen</br>
✅ Modellgüte prüfen</br>
✅ Residuenanalyse erstellen</br>
✅ Feature Importance/Selektion prüfen</br>
✅ Robustheitstest erstellen</br>
✅ Modellinterpretation erstellen</br>
✅ Sensitivitätsanalyse erstellen</br>
✅ Kommunikation (Key Takeaways)</br>

<p><font color='black' size="5">
Prognose
</font></p>

In [ ]:
target_train_pred = model.predict(data_train)
target_test_pred = model.predict(data_test)

<p><font color='black' size="5">
Accuracy
</font></p>

In [ ]:
target_train_pred_ = tf.argmax(target_train_pred, axis=1).numpy()
acc_train = accuracy_score(target_train, target_train_pred_) * 100
print(f"Modell: {model} -- Train -- Accuracy: {acc_train:5.2f}")

In [ ]:
target_test_pred_ = tf.argmax(target_test_pred, axis=1).numpy()
acc_test = accuracy_score(target_test, target_test_pred_) * 100
print(f"Modell: {model} -- Test -- Accuracy: {acc_test:5.2f}")

<p><font color='black' size="5">
Einzelne Vorhersage
</font></p>

In [ ]:
target_test_pred_[0]

In [ ]:
test_index = 0
image = DataFrame(data_test[test_index])

In [ ]:
image.style.set_properties(**{"font-size": "6pt"}).background_gradient("Greys")

In [ ]:
plt.imshow(data_test[test_index].reshape((28, 28)))

# 5 | Deploy
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellexport und -speicherung</br>
✅ Abhängigkeiten und Umgebung</br>
✅ Sicherheit und Datenschutz</br>
✅ In die Produktion integrieren</br>
✅ Tests und Validierung</br>
✅ Dokumentation & Wartung</br>